In [2]:
import re
import pandas as pd

def parse_sdai_log(filepath, output_csv):
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    records = []
    current_record = None

    # Expressões regulares para início de bloco e identificação de Data/Node
    start_pattern = re.compile(r'^(\d+)\s+(\d{2}:\d{2}:\d{2})\s+(.+)')
    date_node_pattern = re.compile(r'^\s*(MON|TUE|WED|THU|FRI|SAT|SUN)\s+(\d{1,2}-[A-Z]{3}-\d{2})\s*(?:\(?(NODE\s+\d+)\)?)?', re.IGNORECASE)

    for line in lines:
        line_clean = line.rstrip('\n')
        # Pular cabeçalhos do TSW e linhas vazias
        if not line_clean.strip() or line_clean.startswith('---'):
            continue
            
        start_match = start_pattern.match(line_clean)
        
        if start_match:
            if current_record:
                records.append(current_record)
            
            # Estrutura base de cada evento
            current_record = {
                "id": start_match.group(1),
                "time": start_match.group(2),
                "description": start_match.group(3).strip(),
                "date": None,
                "node": None,
                "type": None,
                "state": None,
                "raw": [line_clean] # Armazena linhas até resolver a estrutura
            }
        elif current_record:
            current_record["raw"].append(line_clean)
            
    if current_record:
        records.append(current_record)

    # Varredura pós-processamento de cada bloco registrado
    for r in records:
        raw = r["raw"]
        date_idx = -1
        
        for i, l in enumerate(raw):
            match = date_node_pattern.match(l)
            if match:
                date_idx = i
                r["date"] = match.group(2)
                if match.group(3):
                    r["node"] = match.group(3).replace("NODE", "").strip()
                break
                
        # Concatena quebras de linha na descrição principal (ex: descrições que continuam na linha de baixo)
        if date_idx > 1:
            extra_desc = " ".join([raw[i].strip() for i in range(1, date_idx)])
            r["description"] = f"{r['description']} {extra_desc}"
            
        # Extrai Device Type e Status usando o espaçamento em branco do sistema
        if date_idx != -1 and date_idx < len(raw) - 1:
            state_str = " ".join(raw[date_idx+1:]).strip()
            parts = re.split(r'\s{2,}', state_str)
            if len(parts) >= 2:
                r["type"] = parts[0].strip()
                r["state"] = parts[-1].strip()
            elif len(parts) == 1:
                r["type"] = parts[0].strip()
                r["state"] = parts[0].strip()

    # Formatação do DataFrame Final
    df = pd.DataFrame(records).drop(columns=["raw"])
    
    # Padronização de datas para ordenação cronológica
    df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'], format='%d-%b-%y %H:%M:%S', errors='coerce')
    df = df[['id', 'datetime', 'node', 'description', 'type', 'state']]
    
    df.to_csv(output_csv, index=False)
    return df

# Exemplo de execução:


In [3]:
df_sdai = parse_sdai_log('log.txt', 'eventos_sdai_completo.csv')